In [1]:
#cell 1
# Setup imports

import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from google.colab import drive
from IPython.display import display

In [2]:
#cell 2
# Mount Google Drive silently

import io
import contextlib

with contextlib.redirect_stdout(io.StringIO()):
    drive.mount("/content/drive", force_remount=False)

In [3]:
#cell 3
# Define file paths

MODEL_DATASET_FILES = {
    "qwen3.5": {
        "hotpotqa": Path(
            "/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/"
            "hotpot_answers_qwen3_5_9b/qwen_agent_responses.json"
        ),
        "2wikimultihopqa": Path(
            "/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/"
            "wiki_answers_qwen3_5_9b/qwen_agent_responses.json"
        ),
    },
    "gemma4": {
        "hotpotqa": Path(
            "/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/"
            "hotpot_answers_gemma4/qwen_agent_responses.json"
        ),
        "2wikimultihopqa": Path(
            "/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/"
            "wiki_answers_gemma4/qwen_agent_responses.json"
        ),
    },
    "gpt-oss-120b": {
        "hotpotqa": Path(
            "/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/"
            "hotpot_answers_gpt_oss_120b/qwen_agent_responses.json"
        ),
        "2wikimultihopqa": Path(
            "/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/"
            "wiki_answers_gpt_oss_120b/qwen_agent_responses.json"
        ),
    },
}

In [4]:
#cell 4
# Normalize text and tokenize answers

def normalize_text(text):
    """
    Basic normalization for token-level comparison.
    Lowercase, remove punctuation, and normalize spaces.
    """
    if text is None:
        text = ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.casefold()

    chars = []
    for ch in text:
        if unicodedata.category(ch).startswith("P"):
            chars.append(" ")
        else:
            chars.append(ch)

    text = "".join(chars)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    """
    Convert answer text to a set of tokens.
    """
    normalized = normalize_text(text)
    if not normalized:
        return set()
    return set(normalized.split())

In [5]:
#cell 5
# Compute Precision, Recall, and F1 for one example

def compute_token_f1(predicted_answer, ground_truth_answer):
    """
    Compute token-level Precision, Recall, and F1.
    """
    pred_tokens = tokenize(predicted_answer)
    gt_tokens = tokenize(ground_truth_answer)

    if len(pred_tokens) == 0 and len(gt_tokens) == 0:
        return 1.0, 1.0, 1.0

    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return 0.0, 0.0, 0.0

    overlap = pred_tokens.intersection(gt_tokens)

    precision = len(overlap) / len(pred_tokens)
    recall = len(overlap) / len(gt_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (2 * precision * recall) / (precision + recall)

    return precision, recall, f1

In [6]:
#cell 6
# Load one JSON file and compute row-level scores

def load_json_file(file_path):
    """
    Load a JSON answer file.
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"Expected a list of records in: {file_path}")

    return data


def evaluate_dataset(model_name, dataset_name, file_path):
    """
    Evaluate all questions for one model and one dataset.
    """
    data = load_json_file(file_path)
    rows = []

    for idx, item in enumerate(data):
        gt = item.get("gt", "")
        response = item.get("response", "")
        question_type = item.get("type", "unknown")

        precision, recall, f1 = compute_token_f1(
            predicted_answer=response,
            ground_truth_answer=gt
        )

        rows.append({
            "model": model_name,
            "dataset": dataset_name,
            "row_index": idx,
            "source_index": item.get("source_index", idx),
            "type": question_type if question_type is not None else "unknown",
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "gt": gt,
            "response": response,
        })

    return pd.DataFrame(rows)

In [7]:
#cell 7
# Evaluate all models and datasets

def evaluate_all_model_datasets(model_dataset_files):
    """
    Evaluate every model and dataset file.
    """
    all_dfs = []

    for model_name, dataset_files in model_dataset_files.items():
        for dataset_name, file_path in dataset_files.items():
            dataset_df = evaluate_dataset(
                model_name=model_name,
                dataset_name=dataset_name,
                file_path=file_path
            )
            all_dfs.append(dataset_df)

    return pd.concat(all_dfs, ignore_index=True)

In [8]:
#cell 8
# Build overall and per-type summaries

def build_summary(scores_df):
    """
    Create overall and per-type macro F1 summaries.
    """
    overall_df = (
        scores_df
        .groupby(["model", "dataset"], as_index=False)
        .agg(
            n_questions=("f1", "size"),
            precision_macro=("precision", "mean"),
            recall_macro=("recall", "mean"),
            f1_macro=("f1", "mean"),
        )
    )
    overall_df.insert(2, "type", "overall")

    type_df = (
        scores_df
        .groupby(["model", "dataset", "type"], as_index=False)
        .agg(
            n_questions=("f1", "size"),
            precision_macro=("precision", "mean"),
            recall_macro=("recall", "mean"),
            f1_macro=("f1", "mean"),
        )
    )

    summary_df = pd.concat([overall_df, type_df], ignore_index=True)

    summary_df["f1_percent"] = summary_df["f1_macro"] * 100

    model_order = ["qwen3.5", "gemma4", "gpt-oss-120b"]
    dataset_order = ["hotpotqa", "2wikimultihopqa"]

    summary_df["model"] = pd.Categorical(
        summary_df["model"],
        categories=model_order,
        ordered=True
    )

    summary_df["dataset"] = pd.Categorical(
        summary_df["dataset"],
        categories=dataset_order,
        ordered=True
    )

    summary_df["type_sort"] = summary_df["type"].apply(
        lambda x: "000_overall" if x == "overall" else str(x)
    )

    summary_df = (
        summary_df
        .sort_values(["model", "dataset", "type_sort"])
        .drop(columns=["type_sort"])
        .reset_index(drop=True)
    )

    numeric_cols = ["precision_macro", "recall_macro", "f1_macro", "f1_percent"]
    summary_df[numeric_cols] = summary_df[numeric_cols].round(6)

    return summary_df

In [9]:
#cell 9
# Final output only

scores_df = evaluate_all_model_datasets(MODEL_DATASET_FILES)
summary_df = build_summary(scores_df)

display(summary_df)

,model,dataset,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,qwen3.5,hotpotqa,overall,1000,0.326315,0.527558,0.366035,36.603519
1,qwen3.5,hotpotqa,bridge,700,0.356005,0.483084,0.380397,38.039703
2,qwen3.5,hotpotqa,comparison,300,0.257037,0.631330,0.332524,33.252424
3,qwen3.5,2wikimultihopqa,overall,1000,0.248479,0.401854,0.284897,28.489699
4,qwen3.5,2wikimultihopqa,bridge_comparison,250,0.265370,0.389638,0.292709,29.270918
5,qwen3.5,2wikimultihopqa,comparison,250,0.247384,0.635333,0.341207,34.120668
6,qwen3.5,2wikimultihopqa,compositional,250,0.117110,0.178629,0.131607,13.160693
7,qwen3.5,2wikimultihopqa,inference,250,0.364052,0.403814,0.374065,37.406518
8,gemma4,hotpotqa,overall,1000,0.364476,0.392138,0.357374,35.737392
9,gemma4,hotpotqa,bridge,700,0.395310,0.392147,0.381371,38.137091
